# Common · 01 — CirCor parsing, preprocessing and signal-processing library

Requires `common/00_config.ipynb`. Implements Phases 2–7 building blocks (24AIM301 Units 1–2) and evaluation metrics.

## `circor`

CirCor DigiScope v1.0.3 parsing, recording-level labels and patient-independent split.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.model_selection import StratifiedShuffleSplit

LOCATIONS = ["AV", "PV", "TV", "MV", "Phc"]


def parse_subject_txt(path: Path) -> dict:
    lines = path.read_text().strip().splitlines()
    pid, n_rec, fs = lines[0].split()[:3]
    recs = []
    for ln in lines[1 : 1 + int(n_rec)]:
        loc, hea, wav, tsv = ln.split()[:4]
        recs.append({"location": loc, "hea": hea, "wav": wav, "tsv": tsv})
    info = {}
    for ln in lines[1 + int(n_rec):]:
        if ln.startswith("#") and ":" in ln:
            k, v = ln[1:].split(":", 1)
            info[k.strip()] = v.strip()
    return {"patient_id": int(pid), "fs": int(fs), "recordings": recs, "info": info}


def parse_hea(path: Path) -> dict:
    first = path.read_text().splitlines()[0].split()
    return {"fs_hea": int(first[2]), "n_samples": int(first[3])}


def read_tsv(path: Path) -> np.ndarray:
    arr = np.loadtxt(path, ndmin=2)
    return arr if arr.size else np.zeros((0, 3))


def build_records_table(raw_dir: Path) -> pd.DataFrame:
    tdir = raw_dir / "training_data"
    rows = []
    for txt in sorted(tdir.glob("*.txt")):
        subj = parse_subject_txt(txt)
        info = subj["info"]
        murmur_locs = set() if info.get("Murmur locations", "nan") == "nan" else set(info["Murmur locations"].split("+"))
        for r in subj["recordings"]:
            wav = tdir / r["wav"]
            row = {
                "record_id": Path(r["wav"]).stem,
                "patient_id": subj["patient_id"],
                "location": r["location"],
                "wav_path": str(wav.relative_to(raw_dir)),
                "hea_path": str((tdir / r["hea"]).relative_to(raw_dir)),
                "tsv_path": str((tdir / r["tsv"]).relative_to(raw_dir)),
                "exists": wav.exists() and (tdir / r["hea"]).exists() and (tdir / r["tsv"]).exists(),
                "patient_murmur": info.get("Murmur"),
                "outcome": info.get("Outcome"),
                "murmur_locations": info.get("Murmur locations"),
                "most_audible": info.get("Most audible location"),
                "age": info.get("Age"),
                "sex": info.get("Sex"),
                "pregnant": info.get("Pregnancy status"),
                "height": pd.to_numeric(info.get("Height"), errors="coerce"),
                "weight": pd.to_numeric(info.get("Weight"), errors="coerce"),
                "campaign": info.get("Campaign"),
            }
            for k in ["timing", "shape", "grading", "pitch", "quality"]:
                row[f"sys_{k}"] = info.get(f"Systolic murmur {k}")
                row[f"dia_{k}"] = info.get(f"Diastolic murmur {k}")
            if row["patient_murmur"] == "Present":
                row["murmur_label"] = "Present" if r["location"] in murmur_locs else "Absent"
            else:
                row["murmur_label"] = row["patient_murmur"]
            if row["exists"]:
                row.update(parse_hea(tdir / r["hea"]))
                info_sf = sf.info(str(wav))
                row["fs_wav"] = info_sf.samplerate
                row["duration_s"] = info_sf.frames / info_sf.samplerate
                seg = read_tsv(tdir / r["tsv"])
                ann = seg[seg[:, 2] > 0] if len(seg) else seg
                row["annotated_s"] = float((ann[:, 1] - ann[:, 0]).sum()) if len(ann) else 0.0
                row["n_cycles"] = int((seg[:, 2] == 1).sum()) if len(seg) else 0
                row["heart_rate_tsv"] = 60.0 * row["n_cycles"] / row["annotated_s"] if row["annotated_s"] > 0 else np.nan
            rows.append(row)
    df = pd.DataFrame(rows)
    df["y_murmur"] = df["murmur_label"].map({"Present": 1, "Absent": 0})
    df["y_outcome"] = df["outcome"].map({"Abnormal": 1, "Normal": 0})
    return df


def patient_split(df: pd.DataFrame, fractions: dict, seed: int) -> pd.DataFrame:
    pats = df.groupby("patient_id")["patient_murmur"].first().reset_index()
    y = pats["patient_murmur"].values
    idx = np.arange(len(pats))
    test_frac = fractions["test"]
    val_frac = fractions["val"] / (1 - test_frac)
    s1 = StratifiedShuffleSplit(1, test_size=test_frac, random_state=seed)
    trval, test = next(s1.split(idx, y))
    s2 = StratifiedShuffleSplit(1, test_size=val_frac, random_state=seed)
    tr, va = next(s2.split(trval, y[trval]))
    split = np.empty(len(pats), dtype=object)
    split[trval[tr]], split[trval[va]], split[test] = "train", "val", "test"
    pats["split"] = split
    out = df.merge(pats[["patient_id", "split"]], on="patient_id")
    assert out.groupby("patient_id")["split"].nunique().max() == 1, "patient leakage across splits"
    return out

## `signal`

Phase 4-5: quality control and preprocessing (24AIM301 Unit 1: noise, IIR/FIR filters).

In [ ]:
import numpy as np
import soundfile as sf
from scipy import signal as sps


def load_wav(path) -> tuple[np.ndarray, int, dict]:
    raw, fs = sf.read(str(path), dtype="int16")
    raw = raw.astype(np.int32)
    full = np.iinfo(np.int16).max
    qc = {
        "clip_fraction": float(np.mean(np.abs(raw) >= full - 1)),
        "peak_abs": int(np.abs(raw).max()) if raw.size else 0,
    }
    return raw.astype(np.float32) / full, fs, qc


def butter_bandpass_sos(lo: float, hi: float, fs: int, order: int = 4):
    return sps.butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")


def fir_bandpass(lo: float, hi: float, fs: int, numtaps: int = 201):
    return sps.firwin(numtaps, [lo, hi], pass_zero=False, fs=fs, window="hamming")


def bandpass(x: np.ndarray, lo: float, hi: float, fs: int, order: int = 4, kind: str = "iir") -> np.ndarray:
    hi = min(hi, 0.95 * fs / 2)
    if kind == "iir":
        return sps.sosfiltfilt(butter_bandpass_sos(lo, hi, fs, order), x).astype(np.float32)
    taps = fir_bandpass(lo, hi, fs)
    return sps.filtfilt(taps, [1.0], x, padlen=min(len(x) - 1, 3 * len(taps))).astype(np.float32)


def schmidt_spike_removal(x: np.ndarray, fs: int, win_s: float = 0.5, max_iter: int = 200) -> np.ndarray:
    x = x.copy()
    w = int(win_s * fs)
    n = len(x) // w
    if n < 3:
        return x
    frames = x[: n * w].reshape(n, w)
    for _ in range(max_iter):
        maa = np.abs(frames).max(axis=1)
        if not np.any(maa > 3 * np.median(maa)):
            break
        k = int(np.argmax(maa))
        fr = frames[k]
        pos = int(np.argmax(np.abs(fr)))
        zc = np.where(np.diff(np.signbit(fr)))[0]
        before = zc[zc < pos]
        after = zc[zc >= pos]
        a = before[-1] + 1 if len(before) else 0
        b = after[0] + 1 if len(after) else w
        fr[a:b] = 0.0
    return x


def robust_normalize(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    return ((x - med) / (iqr + 1e-8)).astype(np.float32)


def clip_extremes(x: np.ndarray, q: float = 99.9) -> np.ndarray:
    c = np.percentile(np.abs(x), q)
    return np.clip(x, -c, c)


def annotated_span(seg: np.ndarray) -> tuple[float, float] | None:
    ann = seg[seg[:, 2] > 0] if len(seg) else seg
    if not len(ann):
        return None
    return float(ann[0, 0]), float(ann[-1, 1])


def preprocess(raw: np.ndarray, fs_in: int, cfg: dict, seg: np.ndarray | None = None) -> np.ndarray:
    p = cfg["preprocessing"]
    x = raw
    if p.get("trim_to_annotation", False) and seg is not None:
        span = annotated_span(seg)
        if span and span[1] - span[0] >= p["min_duration_s"]:
            x = x[int(span[0] * fs_in) : int(span[1] * fs_in)]
    fs = p["fs"]
    g = np.gcd(fs, fs_in)
    x = sps.resample_poly(x, fs // g, fs_in // g).astype(np.float32)
    x = bandpass(x, *p["band"], fs, p["filter_order"])
    if p.get("spike_removal", True):
        x = schmidt_spike_removal(x, fs)
    return clip_extremes(robust_normalize(x))


def homomorphic_envelope(x: np.ndarray, fs: int, cutoff: float = 8.0) -> np.ndarray:
    a = np.abs(sps.hilbert(x)) + 1e-8
    sos = sps.butter(1, cutoff, btype="low", fs=fs, output="sos")
    return np.exp(sps.sosfiltfilt(sos, np.log(a))).astype(np.float32)


def add_noise(x: np.ndarray, snr_db: float, rng: np.random.Generator, kind: str = "white", fs: int = 2000) -> np.ndarray:
    p_sig = np.mean(x**2) + 1e-12
    if kind == "white":
        n = rng.standard_normal(len(x))
    elif kind == "pink":
        f = np.fft.rfftfreq(len(x), 1 / fs)
        spec = (rng.standard_normal(len(f)) + 1j * rng.standard_normal(len(f))) / np.sqrt(np.maximum(f, 1.0))
        n = np.fft.irfft(spec, len(x))
    elif kind == "impulse":
        n = np.zeros(len(x))
        k = max(1, len(x) // fs)
        for c in rng.integers(0, len(x), k):
            L = int(rng.uniform(0.01, 0.05) * fs)
            n[c : c + L] += rng.standard_normal(min(L, len(x) - c)) * np.hanning(min(L, len(x) - c))
    else:
        raise ValueError(kind)
    n = n * np.sqrt(p_sig / (np.mean(n**2) + 1e-12) / 10 ** (snr_db / 10))
    return (x + n).astype(np.float32)

## `sip`

Classical signal descriptors (24AIM301 Units 1-2): PSD, cepstrum, coherence, spectral shape.

These form the RL *state* (signal context) and the SIP analysis figures.

In [ ]:
import numpy as np
from scipy import signal as sps


def welch_psd(x: np.ndarray, fs: int, nperseg: int = 512):
    return sps.welch(x, fs=fs, nperseg=min(nperseg, len(x)), noverlap=None, window="hann")


def spectral_shape(f: np.ndarray, pxx: np.ndarray) -> dict:
    p = pxx / (pxx.sum() + 1e-12)
    centroid = float((f * p).sum())
    bandwidth = float(np.sqrt(((f - centroid) ** 2 * p).sum()))
    flatness = float(np.exp(np.mean(np.log(pxx + 1e-12))) / (np.mean(pxx) + 1e-12))
    rolloff = float(f[np.searchsorted(np.cumsum(p), 0.85)])
    entropy = float(-(p * np.log2(p + 1e-12)).sum() / np.log2(len(p)))
    return {"centroid": centroid, "bandwidth": bandwidth, "flatness": flatness, "rolloff85": rolloff, "spec_entropy": entropy}


def cepstral_heart_rate(x: np.ndarray, fs: int, bpm_range=(50, 220)) -> tuple[float, float]:
    env = homomorphic_envelope(x, fs)
    fs_e = 50
    env = sps.resample_poly(env, fs_e, fs)
    env = env - env.mean()
    spec = np.abs(np.fft.rfft(env * np.hanning(len(env)), n=4 * len(env))) + 1e-8
    ceps = np.fft.irfft(np.log(spec))
    q = np.arange(len(ceps)) / fs_e
    lo, hi = 60 / bpm_range[1], 60 / bpm_range[0]
    m = (q >= lo) & (q <= hi)
    if not m.any():
        return np.nan, 0.0
    k = np.argmax(ceps[m])
    period = q[m][k]
    prom = float(ceps[m][k] / (np.std(ceps[m]) + 1e-8))
    return 60.0 / period, prom


def autocorr_heart_rate(x: np.ndarray, fs: int, bpm_range=(50, 220)) -> float:
    env = sps.resample_poly(homomorphic_envelope(x, fs), 50, fs)
    env = env - env.mean()
    ac = np.correlate(env, env, "full")[len(env) - 1 :]
    lags = np.arange(len(ac)) / 50
    m = (lags >= 60 / bpm_range[1]) & (lags <= 60 / bpm_range[0])
    return 60.0 / lags[m][np.argmax(ac[m])] if m.any() else np.nan


def snr_estimate_db(x: np.ndarray, fs: int, win_s: float = 0.02) -> float:
    w = int(win_s * fs)
    n = len(x) // w
    e = (x[: n * w].reshape(n, w) ** 2).mean(1) + 1e-12
    return float(10 * np.log10(np.percentile(e, 95) / np.percentile(e, 10)))


def coherence(x: np.ndarray, y: np.ndarray, fs: int, nperseg: int = 512):
    n = min(len(x), len(y))
    return sps.coherence(x[:n], y[:n], fs=fs, nperseg=nperseg)


def context_descriptors(x: np.ndarray, fs: int) -> dict:
    f, pxx = welch_psd(x, fs)
    d = spectral_shape(f, pxx)
    bpm, prom = cepstral_heart_rate(x, fs)
    d.update(
        snr_db=snr_estimate_db(x, fs),
        log_energy=float(np.log(np.mean(x**2) + 1e-12)),
        variance=float(np.var(x)),
        zcr=float(np.mean(np.abs(np.diff(np.signbit(x).astype(np.int8))))),
        duration_s=len(x) / fs,
        kurtosis=float(((x - x.mean()) ** 4).mean() / (x.var() ** 2 + 1e-12)),
        ceps_bpm=bpm,
        ceps_prominence=prom,
        acf_bpm=autocorr_heart_rate(x, fs),
    )
    edges = np.linspace(20, 800, 33)
    logpsd = [np.log(pxx[(f >= a) & (f < b)].mean() + 1e-12) for a, b in zip(edges[:-1], edges[1:])]
    for i, v in enumerate(logpsd):
        d[f"logpsd_{i:02d}"] = float(v)
    return d

## `representations`

Phase 7 candidate time-frequency representations (24AIM301 Unit 2: Fourier & wavelet transforms).

Wavelet branch uses a *wavelet-packet* decomposition in frequency order: at level L the
0..fs/2 axis is split into 2^L equal sub-bands, so the coefficient matrix is a genuine
time-frequency image for any discrete wavelet (db/sym/coif), and the level trades
time resolution against frequency resolution.

In [ ]:
import numpy as np
import pywt
from scipy import signal as sps


def wavelet_packet_matrix(x: np.ndarray, wavelet: str, level: int) -> np.ndarray:
    wp = pywt.WaveletPacket(x, wavelet=wavelet, mode="periodization", maxlevel=level)
    nodes = wp.get_level(level, order="freq")
    return np.stack([n.data for n in nodes]).astype(np.float32)


def band_rows(n_rows: int, band: tuple[float, float], fs: int) -> slice:
    width = (fs / 2) / n_rows
    lo = int(np.floor(band[0] / width))
    hi = int(np.ceil(band[1] / width))
    lo = min(max(lo, 0), n_rows - 1)
    hi = max(min(hi, n_rows), lo + 1)
    return slice(lo, hi)


def stft_power(x: np.ndarray, fs: int, n_fft: int = 128, hop: int = 32) -> tuple[np.ndarray, np.ndarray]:
    f, _, z = sps.stft(x, fs=fs, window="hann", nperseg=n_fft, noverlap=n_fft - hop, boundary=None, padded=False)
    return f, (np.abs(z) ** 2).astype(np.float32)


def log_compress(m: np.ndarray) -> np.ndarray:
    return np.log1p(np.abs(m) * 10.0).astype(np.float32)

## `config_features`

Configuration-specific feature groups used by the fast proxy classifier that pays out RL rewards.

For every recording and every candidate configuration (wavelet, level, band, window) we compute

  W-group : wavelet-packet sub-band features  (depends on wavelet, level, band, window)
  S-group : STFT band features                 (depends on band, window)
  T-group : temporal / envelope features       (depends on window)

The fusion action picks which groups are concatenated (TS, TW, SW, TSW). Window-level
statistics are pooled over the recording (mean / std / floor-to-peak ratios), which is what
makes the window size a real design choice: murmurs fill the gaps between S1 and S2, so the
within-window floor/peak ratio in the murmur band is the key cue and its reliability depends
on how many cardiac cycles a window spans.

In [ ]:
import numpy as np
import pywt


def _frame_cols(n_cols: int, win: int) -> list[slice]:
    win = max(2, min(win, n_cols))
    hop = max(1, win // 2)
    starts = range(0, max(1, n_cols - win + 1), hop)
    return [slice(s, s + win) for s in starts]


def _pool(v: np.ndarray) -> list[float]:
    return [float(np.mean(v)), float(np.std(v))]


def _frames3d(a: np.ndarray, win: int) -> np.ndarray:
    win = max(2, min(win, a.shape[1]))
    hop = max(1, win // 2)
    return np.lib.stride_tricks.sliding_window_view(a, win, axis=1)[:, ::hop]


def wavelet_group(M: np.ndarray, rows: slice, win_cols: int) -> np.ndarray:
    fr = _frames3d(M[rows], win_cols)
    energy = np.mean(fr ** 2, axis=2) + 1e-8
    loge = np.log(energy)
    mag = np.abs(fr)
    q20, q95 = np.percentile(mag, [20, 95], axis=2)
    floor = q20 / (q95 + 1e-8)
    p = energy / energy.sum(0, keepdims=True)
    ent = -(p * np.log(p + 1e-12)).sum(0) / np.log(max(2, p.shape[0]))
    tot = np.log(energy.sum(0))
    return np.concatenate([loge.mean(1), loge.std(1), floor.mean(1),
                           np.array(_pool(ent) + _pool(tot) + [float(np.median(floor))])]).astype(np.float32)


def stft_group(f: np.ndarray, P: np.ndarray, band, win_frames: int) -> np.ndarray:
    m = (f >= band[0]) & (f <= band[1])
    if m.sum() < 2:
        m[np.argsort(np.abs(f - np.mean(band)))[:2]] = True
    Pb, fb = P[m], f[m]
    fr = _frames3d(Pb, win_frames)
    spec = fr.mean(2).T + 1e-12
    p = spec / spec.sum(1, keepdims=True)
    c = (p * fb).sum(1)
    bw = np.sqrt((p * (fb[None] - c[:, None]) ** 2).sum(1))
    flat = np.exp(np.log(spec).mean(1)) / spec.mean(1)
    ent = -(p * np.log(p)).sum(1) / np.log(p.shape[1])
    frame_pow = fr.sum(0) + 1e-12
    q20, q95 = np.percentile(frame_pow, [20, 95], axis=1)
    R = np.stack([np.log(spec.sum(1)), c, bw, flat, ent, q20 / q95], 1)
    prof = np.log(Pb.mean(1) + 1e-12)
    prof8 = np.array([s.mean() for s in np.array_split(prof, 8)]) if len(prof) >= 8 else np.resize(prof, 8)
    return np.concatenate([R.mean(0), R.std(0), prof8 - prof8.mean()]).astype(np.float32)


def temporal_group(x: np.ndarray, env: np.ndarray, fs: int, win: int) -> np.ndarray:
    frames = _frame_cols(len(x), win)
    rows = []
    for fr in frames:
        xs, es = x[fr], env[fr]
        rms = np.sqrt(np.mean(xs**2) + 1e-12)
        zcr = np.mean(np.abs(np.diff(np.signbit(xs).astype(np.int8))))
        kurt = ((xs - xs.mean()) ** 4).mean() / (xs.var() ** 2 + 1e-12)
        floor = np.percentile(es, 20) / (np.percentile(es, 95) + 1e-8)
        ed = es[:: fs // 50] - es[:: fs // 50].mean()
        ac = np.correlate(ed, ed, "full")[len(ed) - 1 :]
        lo, hi = 50 * 60 // 220, min(len(ac), 50 * 60 // 50)
        reg = ac[lo:hi].max() / (ac[0] + 1e-12) if hi > lo else 0.0
        rows.append([np.log(rms), zcr, np.log(kurt), floor, reg])
    R = np.array(rows)
    return np.concatenate([R.mean(0), R.std(0)]).astype(np.float32)


def recording_features(x: np.ndarray, cfg: dict) -> dict:
    fs = cfg["preprocessing"]["fs"]
    A = cfg["actions"]
    out = {}
    env = homomorphic_envelope(x, fs)
    f, P = stft_power(x, fs, cfg["deep"]["n_fft"], cfg["deep"]["hop"])
    hop = cfg["deep"]["hop"]
    for win_s in A["window_s"]:
        win = int(win_s * fs)
        out[("T", win_s)] = temporal_group(x, env, fs, win)
        for bi, band in enumerate(A["band"]):
            out[("S", bi, win_s)] = stft_group(f, P, band, max(2, win // hop))
    maxL = max(A["level"])
    n = (len(x) // 2**maxL) * 2**maxL
    for wv in A["wavelet"]:
        wp = pywt.WaveletPacket(x[:n], wavelet=wv, mode="periodization", maxlevel=maxL)
        for L in A["level"]:
            M = np.stack([nd.data for nd in wp.get_level(L, order="freq")]).astype(np.float32)
            for bi, band in enumerate(A["band"]):
                rows = band_rows(M.shape[0], band, fs)
                for win_s in A["window_s"]:
                    out[("W", wv, L, bi, win_s)] = wavelet_group(M, rows, max(2, int(win_s * fs) // 2**L))
    return out

## `metrics`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score


def binary_metrics(y, p, thr=0.5) -> dict:
    y = np.asarray(y).astype(int)
    p = np.asarray(p, float)
    yh = (p >= thr).astype(int)
    tp = int(((yh == 1) & (y == 1)).sum())
    tn = int(((yh == 0) & (y == 0)).sum())
    fp = int(((yh == 1) & (y == 0)).sum())
    fn = int(((yh == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    two = len(np.unique(y)) == 2
    return {
        "macro_f1": float(f1_score(y, yh, average="macro", zero_division=0)),
        "sensitivity": sens,
        "specificity": spec,
        "balanced_acc": (sens + spec) / 2,
        "auroc": float(roc_auc_score(y, p)) if two else np.nan,
        "auprc": float(average_precision_score(y, p)) if two else np.nan,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def patient_level(df: pd.DataFrame, prob_col="prob", y_col="patient_y") -> tuple[np.ndarray, np.ndarray]:
    g = df.groupby("patient_id").agg(p=(prob_col, "max"), y=(y_col, "first"))
    return g.y.values, g.p.values


def mean_std_table(rows: list[dict], group="model", cols=("macro_f1", "sensitivity", "specificity", "balanced_acc", "auroc", "auprc")):
    d = pd.DataFrame(rows)
    agg = d.groupby(group, sort=False)[list(cols)].agg(["mean", "std"])
    out = pd.DataFrame(index=agg.index)
    for c in cols:
        out[c] = [f"{m:.3f} Â± {s:.3f}" if np.isfinite(s) else f"{m:.3f}" for m, s in zip(agg[(c, "mean")], agg[(c, "std")])]
    return out

In [ ]:
print("loaded 01_signal_lib")